# Stage 11 — DeepScoresV2 Dense Held-out Evaluation

Final **non-tuning** evaluation of the frozen `best.pt` Residual U-Net checkpoint on the official DeepScoresV2 Dense test split (352 images).
This notebook does not create an optimizer, does not backpropagate, does not update weights, and does not authorize production use.
It compares degraded-input baseline metrics against restored-output metrics and writes `heldout_final_evidence.json` to Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, tarfile, random, math, io, time

A=Path("/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_DATA/ds2_dense.tar.gz")
O=Path("/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_TRAINING_OUTPUT/deepscoresv2_dense_residual_unet_v1")
BEST=O/"best.pt"
X=Path("/content/st_score_restore_deepscoresv2_dense")
MD5="7237318e381e6e0848ec30eb82decb83"
SIZE=741814529
EXPECTED_BEST_SHA256="08b279161a9e8c4bd37376da221ecb4e07130724254ccf7d9591c8d32f368683"
EXPECTED_CONFIG="deff0f1270009839e234608dd9967038e1228a6b2b034e26059ac2f8cbfd0f80"
OFFICIAL_HELD_OUT_IMAGES=352

if not A.exists():
    matches=list(Path("/content/drive/MyDrive").rglob("ds2_dense.tar.gz"))
    if len(matches)!=1:
        raise FileNotFoundError(f"Expected one ds2_dense.tar.gz, found {len(matches)}")
    A=matches[0]
if not BEST.exists():
    raise FileNotFoundError(BEST)
if A.stat().st_size!=SIZE:
    raise RuntimeError("Archive size mismatch")

def file_sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        while b:=f.read(8<<20):
            h.update(b)
    return h.hexdigest()

h=hashlib.md5()
with A.open("rb") as f:
    while b:=f.read(8<<20):
        h.update(b)
if h.hexdigest()!=MD5:
    raise RuntimeError(f"MD5 mismatch: {h.hexdigest()} != {MD5}")
if file_sha256(BEST)!=EXPECTED_BEST_SHA256:
    raise RuntimeError("best.pt SHA256 mismatch")

def safe_extract():
    X.mkdir(parents=True,exist_ok=True)
    base=X.resolve()
    with tarfile.open(A,"r:gz") as t:
        for m in t.getmembers():
            p=m.name.replace("\\","/")
            if not p or p.startswith("/") or ".." in Path(p).parts or m.issym() or m.islnk() or m.isdev():
                raise RuntimeError(f"unsafe tar member: {m.name}")
            q=(X/p).resolve()
            if q!=base and base not in q.parents:
                raise RuntimeError(f"tar escape: {m.name}")
        t.extractall(X)

if not X.exists() or not any(X.iterdir()):
    safe_extract()

te=list(X.rglob("deepscores_test.json"))
if len(te)!=1:
    raise RuntimeError("Expected one deepscores_test.json")
R=te[0].parent
payload=json.loads(te[0].read_text())
if not isinstance(payload,dict) or not isinstance(payload.get("images"),list):
    raise RuntimeError("Unsupported test annotation JSON")

H=[]
for row in payload["images"]:
    n=row.get("file_name") or row.get("filename") or row.get("img_name")
    if n:
        H.append(str(n))
if len(H)!=OFFICIAL_HELD_OUT_IMAGES:
    raise RuntimeError(f"Expected {OFFICIAL_HELD_OUT_IMAGES} official held-out images, got {len(H)}")

def image_path(name):
    for p in (R/"images"/name,R/name):
        if p.exists():
            return p
    matches=list(R.rglob(Path(name).name))
    if len(matches)==1:
        return matches[0]
    raise FileNotFoundError(name)

print("Archive + checkpoint identity verified. Official held-out images:",len(H))

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
from torch.utils.data import Dataset,DataLoader
from PIL import Image,ImageFilter
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

if not torch.cuda.is_available():
    raise RuntimeError("GPU required: Colab Runtime > Change runtime type > GPU")

D=torch.device("cuda")
SEED=20260907
PATCH=512
BATCH=4
HELD_OUT_VARIANTS=2

def rng(name,variant):
    return random.Random(int.from_bytes(hashlib.sha256(f"{SEED}:heldout:{name}:{variant}".encode()).digest()[:8],"big"))

def crop(im,r):
    w,h=im.size
    if min(w,h)<PATCH:
        s=max(PATCH/w,PATCH/h)
        im=im.resize((math.ceil(w*s),math.ceil(h*s)),Image.Resampling.BICUBIC)
        w,h=im.size
    a=np.asarray(im)
    best=(0,0); best_ink=-1
    for _ in range(8):
        x=r.randint(0,max(0,w-PATCH)); y=r.randint(0,max(0,h-PATCH))
        ink=(a[y:y+PATCH,x:x+PATCH]<235).mean()
        if ink>best_ink:
            best_ink=ink; best=(x,y)
    x,y=best
    return im.crop((x,y,x+PATCH,y+PATCH))

def degrade(im,r):
    im=TF.rotate(im,r.uniform(-2.5,2.5),interpolation=InterpolationMode.BILINEAR,fill=255)
    w,h=im.size
    d=r.uniform(.002,.02); dx,dy=int(w*d),int(h*d)
    start=[[0,0],[w-1,0],[w-1,h-1],[0,h-1]]
    end=[[r.randint(0,dx),r.randint(0,dy)],
         [w-1-r.randint(0,dx),r.randint(0,dy)],
         [w-1-r.randint(0,dx),h-1-r.randint(0,dy)],
         [r.randint(0,dx),h-1-r.randint(0,dy)]]
    im=TF.perspective(im,start,end,interpolation=InterpolationMode.BILINEAR,fill=255)
    im=TF.adjust_brightness(im,r.uniform(.78,1.2))
    im=TF.adjust_contrast(im,r.uniform(.82,1.18))
    im=im.filter(ImageFilter.GaussianBlur(r.uniform(.15,1.75)))
    a=np.asarray(im).astype(np.float32)/255.
    yy,xx=np.mgrid[:a.shape[0],:a.shape[1]]
    cx,cy=r.uniform(0,a.shape[1]),r.uniform(0,a.shape[0])
    z=np.sqrt((xx-cx)**2+(yy-cy)**2); z/=max(z.max(),1)
    a*=1-r.uniform(0,.22)*(1-z)
    a+=np.random.default_rng(r.randrange(2**32)).normal(0,r.uniform(.002,.03),a.shape)
    a=np.clip(a,0,1)
    im=Image.fromarray((a*255).astype("uint8"))
    b=io.BytesIO(); im.save(b,"JPEG",quality=r.randint(52,95)); b.seek(0)
    return Image.open(b).convert("L")

def ten(im):
    return torch.from_numpy(np.asarray(im,dtype=np.float32)/255.).unsqueeze(0)

class HeldOutPairs(Dataset):
    def __init__(self,names,variants):
        self.names=list(names); self.variants=variants
    def __len__(self):
        return len(self.names)*self.variants
    def __getitem__(self,i):
        name=self.names[i//self.variants]
        r=rng(name,i%self.variants)
        target=crop(Image.open(image_path(name)).convert("L"),r)
        source=degrade(target,r)
        return ten(source),ten(target),name

loader=DataLoader(HeldOutPairs(H,HELD_OUT_VARIANTS),BATCH,shuffle=False,num_workers=2,pin_memory=True)

class C(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.n=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.GroupNorm(8,b),nn.SiLU(),
                             nn.Conv2d(b,b,3,padding=1),nn.GroupNorm(8,b),nn.SiLU())
    def forward(self,x):
        return self.n(x)

class UNet(nn.Module):
    def __init__(self,b=32):
        super().__init__()
        self.e1=C(1,b); self.e2=C(b,2*b); self.e3=C(2*b,4*b); self.mid=C(4*b,8*b)
        self.d3=C(12*b,4*b); self.d2=C(6*b,2*b); self.d1=C(3*b,b); self.o=nn.Conv2d(b,1,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(F.max_pool2d(a,2)); c=self.e3(F.max_pool2d(b,2))
        d=self.mid(F.max_pool2d(c,2))
        d=self.d3(torch.cat([F.interpolate(d,size=c.shape[-2:],mode="bilinear",align_corners=False),c],1))
        d=self.d2(torch.cat([F.interpolate(d,size=b.shape[-2:],mode="bilinear",align_corners=False),b],1))
        d=self.d1(torch.cat([F.interpolate(d,size=a.shape[-2:],mode="bilinear",align_corners=False),a],1))
        return torch.clamp(x+torch.tanh(self.o(d))*.5,0,1)

m=UNet().to(D)
ck=torch.load(BEST,map_location=D)
if ck.get("md5")!=MD5 or ck.get("cfg")!=EXPECTED_CONFIG or ck.get("epoch")!=19:
    raise RuntimeError("Checkpoint identity mismatch")
m.load_state_dict(ck["model"])
m.eval()

sx=torch.tensor([[-1.,0,1],[-2,0,2],[-1,0,1]],device=D).view(1,1,3,3)
sy=sx.transpose(2,3)
def edge(x):
    return torch.sqrt(F.conv2d(x,sx,padding=1)**2+F.conv2d(x,sy,padding=1)**2+1e-6)

def state_sha256(model):
    h=hashlib.sha256()
    for name,t in sorted(model.state_dict().items()):
        h.update(name.encode())
        h.update(t.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

WEIGHTS_BEFORE=state_sha256(m)
print("GPU:",torch.cuda.get_device_name(0),"checkpoint epoch:",ck["epoch"])

In [ ]:
def aggregate_pair(pred,target):
    pixel=F.l1_loss(pred,target).item()
    edge_loss=F.l1_loss(edge(pred),edge(target)).item()
    mse=F.mse_loss(pred,target).item()
    total=pixel+.3*edge_loss
    psnr=10*math.log10(1/max(mse,1e-12))
    return np.array([total,pixel,edge_loss,mse,psnr],dtype=np.float64)

base_sum=np.zeros(5,dtype=np.float64)
rest_sum=np.zeros(5,dtype=np.float64)
count=0

with torch.no_grad():
    for x,y,_ in loader:
        x=x.to(D,non_blocking=True); y=y.to(D,non_blocking=True)
        with torch.autocast("cuda",dtype=torch.float16):
            p=m(x)
        n=x.size(0)
        base_sum+=aggregate_pair(x,y)*n
        rest_sum+=aggregate_pair(p,y)*n
        count+=n

baseline=base_sum/count
restored=rest_sum/count
WEIGHTS_AFTER=state_sha256(m)
weights_mutated=WEIGHTS_AFTER!=WEIGHTS_BEFORE

def metric_dict(v):
    return {
        "loss":float(v[0]),
        "pixelL1":float(v[1]),
        "edgeLoss":float(v[2]),
        "mse":float(v[3]),
        "psnrDb":float(v[4]),
    }

baseline_metrics=metric_dict(baseline)
restored_metrics=metric_dict(restored)
quality_improved=(
    restored_metrics["loss"] < baseline_metrics["loss"]
    and restored_metrics["pixelL1"] < baseline_metrics["pixelL1"]
    and restored_metrics["edgeLoss"] < baseline_metrics["edgeLoss"]
    and restored_metrics["psnrDb"] > baseline_metrics["psnrDb"]
)
heldout_pass=quality_improved and not weights_mutated

evidence={
    "artifactType":"stage11_deepscoresv2_dense_heldout_final_evidence",
    "datasetId":"deepscoresv2.dense.v2",
    "archiveMd5":MD5,
    "archiveChecksumVerified":True,
    "checkpointFile":"best.pt",
    "checkpointSha256":EXPECTED_BEST_SHA256,
    "checkpointConfigSha256":EXPECTED_CONFIG,
    "checkpointEpoch":19,
    "gpu":torch.cuda.get_device_name(0),
    "officialHeldOutImages":OFFICIAL_HELD_OUT_IMAGES,
    "variantsPerHeldOutImage":HELD_OUT_VARIANTS,
    "evaluatedPairs":count,
    "optimizerCreated":False,
    "backpropagationExecuted":False,
    "heldOutUsedForTraining":False,
    "heldOutUsedForTuning":False,
    "weightsBeforeSha256":WEIGHTS_BEFORE,
    "weightsAfterSha256":WEIGHTS_AFTER,
    "weightsMutated":weights_mutated,
    "baselineDegraded":baseline_metrics,
    "restored":restored_metrics,
    "improvement":{
        "lossReduction":baseline_metrics["loss"]-restored_metrics["loss"],
        "pixelL1Reduction":baseline_metrics["pixelL1"]-restored_metrics["pixelL1"],
        "edgeLossReduction":baseline_metrics["edgeLoss"]-restored_metrics["edgeLoss"],
        "psnrGainDb":restored_metrics["psnrDb"]-baseline_metrics["psnrDb"],
    },
    "imageQualityGatePass":quality_improved,
    "heldOutEvaluationPass":heldout_pass,
    "stage9aPreservationEvaluationCompleted":False,
    "finalStage11Pass":False,
    "productionInferenceAuthorized":False,
    "modelPublicationAuthorized":False,
    "stage12EntryAuthorized":False,
    "completedAtUnix":int(time.time()),
}
(O/"heldout_final_evidence.json").write_text(json.dumps(evidence,indent=2))
print(json.dumps(evidence,indent=2))
if not heldout_pass:
    raise RuntimeError("Held-out image-quality gate did not pass; do not tune on held-out. Return to development data with a new experiment identity.")